In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib.animation as animation

from numba import njit

## Representation of the equations:

Suppose there are $n_s$ masses, in a $n_d$ dimensional space, we define the position matrix as $$q\in\mathbb{R}^{n_s \times n_d}$$
For a single mass, the position in space is represented with $$\bold q_i \in \mathbb{R}^{n_d}$$

The entire time series of the positions can be given as (for $N_i$ iterations):
$$q(t) \in \mathbb{R}^{(N_i+1) \times n_s \times n_d} $$

## Force Equation:

The acceleration of mass $i$ due to mass $j$ is given by the gravitational acceleration:

$$\ddot{\bold q}_i = \frac{\bold q_j - \bold q_i}{\left\vert\bold q_j - \bold q_i\right\vert^3}$$
Where we are working in units where $G = m =1$

In [ ]:
@njit
def symplectic_stromer_verlet(func, q0, v0, dt, N_i):
    # number of stars
    n_s = q0.shape[0]
    # number of dimensions (mostly 2, but supports 3)
    n_d = q0.shape[1]

    # allocate arrays to store the entire 
    q = np.empty((N_i + 1, n_s, n_d))
    v = np.empty((N_i + 1, n_s, n_d))
    t = dt * np.arange(N_i + 1)

    # set the initial states
    q[0] = q0
    v[0] = v0

    # main iteration loop
    for i in range(N_i):
        # calculate intermediate variables
        Q1 = q[i] + 0.5 * dt * v[i]
        f1 = func(Q1, t[i] + 0.5 * dt)

        # updation step
        v[i+1] = v[i] + dt * f1
        q[i+1] = q[i] + dt * v[i] + 0.5 * (dt ** 2) * f1

    # return the calculated parameters
    return t, q, v

In [ ]:
@njit
def acceleration(q, t):
    # q is current positions: shape (n_s, n_d)
    
    # pairwise displacement vectors: shape (n_s, n_s, n_d)
    disp = q[:, None, :] - q[None, :, :]

    # pairwise distances: shape (n_s, n_s)
    dist_sq = np.sum(disp ** 2, axis = -1)

    # adds a diagonal term to distances to prevent division by zero
    # doesn't affect the physics, since mass has zero force on itself
    dist_sq = dist_sq + np.identity(q.shape[0])

    # force on j-th mass, due the the i-th mass: (n_s, n_s, n_d)
    force_ij = disp / (dist_sq[:, :, None] ** 1.5)

    # total force on the i-th mass
    force = np.sum(force_ij, axis=0)

    return force

## The actual system

The system we are going to simulate is Moore's 3-body figure eight.

It is a very stable system of gravitationally attracting bodies, chasing each other periodically.

In [ ]:
q0 = np.array([
    [0.97000436, -0.24308753],
    [-0.97000436, 0.24308753],
    [0, 0]
])

v0 = np.array([
    [0.466203685, 0.43236573],
    [0.466203685, 0.43236573],
    [-0.93240737, -0.86473146]
])

In [ ]:
t, q, v = symplectic_stromer_verlet(acceleration, q0, v0, 0.01, 1000)